# SK하이닉스 / S&P 500 주가 트렌드 분석 (2025~2026)

> AI HBM 수요 폭증 시대, SK하이닉스와 S&P500 비교 분석

**데이터 출처**: Yahoo Finance (yfinance 라이브러리)


## 0. 환경 설정

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import STL
import warnings, os
warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
os.makedirs("data", exist_ok=True)
os.makedirs("images", exist_ok=True)
COLOR_HYX, COLOR_SP = "#E8642A", "#2A7AE8"
print("환경 설정 완료")

## 1. 데이터 수집

In [ ]:
START, END = "2025-01-01", "2026-08-23"
hyx = yf.download("000660.KS", start=START, end=END, auto_adjust=True, progress=False)
sp  = yf.download("^GSPC",     start=START, end=END, auto_adjust=True, progress=False)
if isinstance(hyx.columns, pd.MultiIndex): hyx.columns = hyx.columns.get_level_values(0)
if isinstance(sp.columns,  pd.MultiIndex): sp.columns  = sp.columns.get_level_values(0)
hyx_close = hyx["Close"].dropna()
sp_close  = sp["Close"].dropna()
hyx.to_csv("data/skhynix_2025_2026.csv")
sp.to_csv("data/sp500_2025_2026.csv")
print("SK하이닉스:", len(hyx_close), "거래일")
print("S&P 500   :", len(sp_close),  "거래일")

## 2. 데이터 탐색 — 결측치 확인

In [ ]:
print(hyx[["Close","Volume"]].describe())
print("결측치:", hyx.isnull().sum().sum())
print(sp[["Close","Volume"]].describe())
print("결측치:", sp.isnull().sum().sum())

## 3. 파생 지표 계산

- 정규화 가격 (2025-01-02 = 100)
- 이동평균 (20일, 60일)
- 일일 변화율
- 월별 수익률 집계


In [ ]:
hyx_norm = hyx_close / hyx_close.iloc[0] * 100
sp_norm  = sp_close  / sp_close.iloc[0]  * 100
hyx_ma20 = hyx_close.rolling(20).mean()
hyx_ma60 = hyx_close.rolling(60).mean()
hyx_ret  = hyx_close.pct_change() * 100
sp_ret   = sp_close.pct_change()  * 100
hyx_monthly = hyx_ret.resample("ME").sum()
sp_monthly  = sp_ret.resample("ME").sum()
print("파생 지표 계산 완료")

## 4. 시각화 1 — 정규화 가격 추이 비교 (필수)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(hyx_norm.index, hyx_norm.values, color=COLOR_HYX, linewidth=1.8, label="SK하이닉스")
ax.plot(sp_norm.index,  sp_norm.values,  color=COLOR_SP,  linewidth=1.8, label="S&P 500")
ax.axhline(100, color="#888", linewidth=0.8, linestyle="--", label="기준선(=100)")
ax.fill_between(hyx_norm.index, hyx_norm.values, 100, where=(hyx_norm.values>=100), alpha=0.08, color=COLOR_HYX)
ax.fill_between(hyx_norm.index, hyx_norm.values, 100, where=(hyx_norm.values<100),  alpha=0.08, color="red")
ax.set_title("SK하이닉스 vs S&P 500 정규화 가격 추이", fontsize=14, fontweight="bold")
ax.set_xlabel("날짜")
ax.set_ylabel("정규화 지수 (기준=100)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("images/01_price_trend.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. 시각화 2 — SK하이닉스 이동평균 (필수)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(hyx_close.index, hyx_close.values, color="#C0C0C0", linewidth=1.0, alpha=0.7, label="종가")
ax.plot(hyx_ma20.index,  hyx_ma20.values,  color=COLOR_HYX, linewidth=2.0, label="20일 이동평균")
ax.plot(hyx_ma60.index,  hyx_ma60.values,  color="#F0D060", linewidth=2.0, label="60일 이동평균")
cross = (hyx_ma20 > hyx_ma60) & (hyx_ma20.shift(1) <= hyx_ma60.shift(1))
dead  = (hyx_ma20 < hyx_ma60) & (hyx_ma20.shift(1) >= hyx_ma60.shift(1))
for d in hyx_close.index[cross]: ax.axvline(d, color="#50FF80", linewidth=1.2, alpha=0.7, linestyle=":")
for d in hyx_close.index[dead]:  ax.axvline(d, color="#FF5050", linewidth=1.2, alpha=0.7, linestyle=":")
ax.set_title("SK하이닉스 종가 및 이동평균 (20일·60일)", fontsize=14, fontweight="bold")
ax.set_xlabel("날짜")
ax.set_ylabel("주가 (KRW)")
fmt = plt.FuncFormatter(lambda x,p: "{:,.0f}".format(x))
ax.yaxis.set_major_formatter(fmt)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("images/02_moving_average.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. 시각화 3 — 월별 수익률 히트맵 (권장)

In [ ]:
month_labels = ["1월","2월","3월","4월","5월","6월","7월","8월","9월","10월","11월","12월"]
def make_pivot(s):
    df = pd.DataFrame({"year":s.index.year,"month":s.index.month,"ret":s.values})
    p  = df.pivot(index="year", columns="month", values="ret")
    p.columns = [month_labels[c-1] for c in p.columns]
    return p
hyx_pivot = make_pivot(hyx_monthly)
sp_pivot  = make_pivot(sp_monthly)
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for ax, piv, ttl in zip(axes,[hyx_pivot,sp_pivot],["SK하이닉스 월별 수익률(%)","S&P 500 월별 수익률(%)"]):
    sns.heatmap(piv, ax=ax, cmap="RdYlGn", center=0, annot=True, fmt=".1f",
                linewidths=0.5, cbar_kws={"label":"수익률(%)","shrink":0.8})
    ax.set_title(ttl, fontsize=13, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("연도")
plt.suptitle("월별 수익률 히트맵 (2025~2026)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("images/03_monthly_return.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. 시각화 4 — 시계열 분해 STL (보너스)

In [ ]:
stl = STL(hyx_close, period=20, robust=True)
res = stl.fit()
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
pairs = [
    (hyx_close,                                      "원본 종가",         COLOR_HYX),
    (pd.Series(res.trend,    index=hyx_close.index), "추세 (Trend)",     "#F0D060"),
    (pd.Series(res.seasonal, index=hyx_close.index), "계절성 (Seasonal)","#60D0F0"),
    (pd.Series(res.resid,    index=hyx_close.index), "잔차 (Residual)",  "#A0A0A0"),
]
for ax,(s,lbl,col) in zip(axes,pairs):
    ax.plot(s.index, s.values, color=col, linewidth=1.5)
    ax.set_ylabel(lbl, fontsize=10)
    ax.grid(True, alpha=0.2)
    if "Residual" in lbl: ax.axhline(0, color="#888", linewidth=0.8, linestyle="--")
axes[0].set_title("SK하이닉스 시계열 분해 (STL, period=20거래일)", fontsize=14, fontweight="bold")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("images/04_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. 통계 요약

In [ ]:
rows = {
    "시작가":       [str(round(hyx_close.iloc[0],0))+" KRW", str(round(sp_close.iloc[0],2))],
    "현재가":       [str(round(hyx_close.iloc[-1],0))+" KRW", str(round(sp_close.iloc[-1],2))],
    "누적수익률":   [str(round((hyx_close.iloc[-1]/hyx_close.iloc[0]-1)*100,1))+"%", str(round((sp_close.iloc[-1]/sp_close.iloc[0]-1)*100,1))+"%"],
    "MDD":         [str(round(((hyx_close/hyx_close.cummax())-1).min()*100,1))+"%", str(round(((sp_close/sp_close.cummax())-1).min()*100,1))+"%"],
    "일평균수익률": [str(round(hyx_ret.mean(),3))+"%", str(round(sp_ret.mean(),3))+"%"],
    "일간std":     [str(round(hyx_ret.std(),2))+"%",  str(round(sp_ret.std(),2))+"%"],
    "최대상승":    [str(round(hyx_ret.max(),2))+"%",  str(round(sp_ret.max(),2))+"%"],
    "최대하락":    [str(round(hyx_ret.min(),2))+"%",  str(round(sp_ret.min(),2))+"%"],
}
summary = pd.DataFrame(rows, index=["SK하이닉스", "S&P 500"]).T
print(summary)